# sweet.tv: temporal recommendation experiment
Elizaveta Gill. Code prepared with Codex; student verification is recorded separately.
CPU only. No credentials, raw inputs or test viewing behaviour are embedded.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["OMP_NUM_THREADS"]="2"


In [ ]:
from pathlib import Path
Path('core.py').write_text('"""Auditable pure-Python metric, interval join and submission contract.\n\nNo competition data or credentials are embedded. Times must be seconds on a\nsingle, verified axis. All intervals use half-open [start, end) semantics.\n"""\nfrom bisect import bisect_left, bisect_right\nfrom collections import defaultdict\nimport csv\nimport math\n\n\ndef ap_at_k(actual, predicted, k=5):\n    """Kaggle-style AP@k; empty truth is 0, duplicates get no second credit."""\n    if k < 1:\n        raise ValueError(\'k must be positive\')\n    actual = set(actual)\n    if not actual:\n        return 0.0\n    seen, hits, total = set(), 0, 0.0\n    for rank, item in enumerate(predicted[:k], 1):\n        if item in actual and item not in seen:\n            hits += 1\n            total += hits / rank\n        seen.add(item)\n    return total / min(k, len(actual))\n\n\ndef map_at_k(truth, predictions, users, k=5):\n    """Fixed evaluation cohort, INCLUDING users without relevant validation items."""\n    users = list(users)\n    if not users or len(users) != len(set(users)):\n        raise ValueError(\'Evaluation users must be unique and nonempty\')\n    if set(predictions) != set(users):\n        raise ValueError(\'Predictions must cover the exact evaluation cohort\')\n    return sum(ap_at_k(truth.get(u, ()), predictions[u], k) for u in users) / len(users)\n\n\ndef union_length(intervals):\n    """Seconds watched without double counting overlapping view-log intervals."""\n    end, total = None, 0\n    for left, right in sorted(intervals):\n        if right <= left:\n            continue\n        if end is None or left > end:\n            total += right-left\n        elif right > end:\n            total += right-end\n        end = right if end is None else max(end, right)\n    return total\n\n\nclass ScheduleIndex:\n    """Supports overlapping broadcasts, not just a single as-of match.\n\n    rows: (channel, start_seconds, end_seconds, nonzero_item_id, airing_key).\n    Prefix maximum ends allow efficient removal of all broadcasts ending before\n    a viewing starts, even when the schedule itself contains overlapping slots.\n    """\n    def __init__(self, rows):\n        by_channel = defaultdict(list)\n        for channel, start, end, item, key in rows:\n            if end <= start:\n                raise ValueError(\'Broadcast length must be positive\')\n            if item != 0:\n                by_channel[channel].append((start, end, item, key))\n        self.channels = {}\n        for channel, values in by_channel.items():\n            values.sort()\n            starts, prefix, largest = [], [], -math.inf\n            for start, end, _, _ in values:\n                starts.append(start)\n                largest=max(largest, end)\n                prefix.append(largest)\n            self.channels[channel]=(values, starts, prefix)\n\n    def overlaps(self, channel, start, end):\n        if end <= start:\n            return []\n        if channel not in self.channels:\n            return []\n        values, starts, prefix = self.channels[channel]\n        lo=bisect_right(prefix, start)\n        hi=bisect_left(starts, end)\n        result=[]\n        for bstart, bend, item, key in values[lo:hi]:\n            left, right=max(start,bstart), min(end,bend)\n            if right > left:\n                result.append((item,key,left,right,bend-bstart))\n        return result\n\n\ndef aggregate_watches(index, views, window_start, window_end, min_seconds):\n    """Clip at split BEFORE matching and merge duplicate watches per airing.\n\n    views: (user,channel,start,end). Returns qualified per-airing interactions.\n    The duration threshold is a parameter; its final value requires data audit.\n    """\n    if window_end <= window_start or min_seconds <= 0:\n        raise ValueError(\'Invalid window or threshold\')\n    segments=defaultdict(list)\n    for user,channel,start,end in views:\n        left,right=max(start,window_start),min(end,window_end)\n        for item,key,a,b,length in index.overlaps(channel,left,right):\n            segments[(user,item,key,length)].append((a,b))\n    results=[]\n    for (user,item,key,length),intervals in segments.items():\n        seconds=union_length(intervals)\n        if seconds >= min_seconds:\n            results.append({\'user_id\':user,\'tv_show_id\':item,\'airing_key\':key,\n                            \'watch_seconds\':seconds,\'programme_seconds\':length,\n                            \'watch_fraction\':seconds/length})\n    return results\n\n\ndef validate_submission(template_path, submission_path, candidates):\n    """Use the ACTUAL template\'s header/order, never an assumed column name."""\n    def read(path):\n        with open(path,newline=\'\',encoding=\'utf-8-sig\') as stream:\n            rows=list(csv.reader(stream))\n        return rows[0],rows[1:]\n    header,template=read(template_path)\n    out_header,rows=read(submission_path)\n    if out_header != header or len(header)!=2 or len(rows)!=len(template):\n        raise ValueError(\'Header or row count differs from real template\')\n    if any(len(r)!=2 for r in rows):\n        raise ValueError(\'Unexpected CSV field count\')\n    if [r[0] for r in rows] != [r[0] for r in template]:\n        raise ValueError(\'User identifiers/order differ from real template\')\n    if len({r[0] for r in rows}) != len(rows):\n        raise ValueError(\'Duplicate users\')\n    allowed=set(map(int,candidates))-{0}\n    for user,ranking in rows:\n        ids=[int(x) for x in ranking.split()]\n        if len(ids)!=5 or len(set(ids))!=5 or not set(ids)<=allowed:\n            raise ValueError(f\'Invalid ranking for user {user}\')\n    return {\'users\':len(rows),\'header\':header,\'five_unique_valid_ids\':True}\n')


In [ ]:
from pathlib import Path
Path('als.py').write_text('"""Weighted implicit ALS (Hu, Koren & Volinsky), exact normal-equation solver.\n\np_ui=1 if observed, else 0. c_ui=1+alpha*log1p(n_ui) if observed,\nelse 1. Minimize sum c_ui*(p_ui-x_u@y_i)^2 + reg*(||X||²+||Y||²).\nNo unobserved pair is materialized. The missing-data baseline is Y.T@Y.\n"""\nimport numpy as np\nfrom numba import njit,prange,set_num_threads\nfrom threadpoolctl import threadpool_limits\n\n@njit(parallel=True,cache=False)\ndef solve_side(indptr,indices,confidence,fixed,regularization):\n    factors=fixed.shape[1]\n    gram=fixed.T@fixed\n    result=np.zeros((len(indptr)-1,factors),np.float64)\n    for row in prange(len(indptr)-1):\n        lo,hi=indptr[row],indptr[row+1]\n        if lo==hi:\n            continue\n        a=gram.copy()\n        b=np.zeros(factors,np.float64)\n        for j in range(factors): a[j,j]+=regularization\n        for ptr in range(lo,hi):\n            idx=indices[ptr];c=confidence[ptr]\n            for j in range(factors):\n                b[j]+=c*fixed[idx,j]\n                for k in range(factors):\n                    a[j,k]+=(c-1)*fixed[idx,j]*fixed[idx,k]\n        result[row]=np.linalg.solve(a,b)\n    return result\n\nclass ImplicitALS:\n    def __init__(self,factors=32,regularization=.1,iterations=20,random_state=2026,num_threads=2):\n        self.factors=factors;self.regularization=regularization\n        self.iterations=iterations;self.random_state=random_state;self.num_threads=num_threads\n\n    def fit(self,confidence):\n        set_num_threads(self.num_threads)\n        csr=confidence.tocsr().astype(np.float64);csc=csr.T.tocsr()\n        rng=np.random.default_rng(self.random_state)\n        self.item_factors=rng.normal(0,.01,(csr.shape[1],self.factors))\n        with threadpool_limits(limits=1):\n            for iteration in range(self.iterations):\n                self.user_factors=solve_side(csr.indptr,csr.indices,csr.data,self.item_factors,self.regularization)\n                self.item_factors=solve_side(csc.indptr,csc.indices,csc.data,self.user_factors,self.regularization)\n        return self\n\ndef verify_solver():\n    """Independent dense objective verifies one update and cold-user behavior."""\n    from scipy.sparse import csr_matrix\n    fixed=np.array([[.1,.2],[.3,-.2],[.5,.4]],dtype=np.float64)\n    conf=csr_matrix([[3.,0.,5.],[0.,0.,0.]])\n    actual=solve_side(conf.indptr,conf.indices,conf.data,fixed,.1)\n    c=np.array([3.,1.,5.]);p=np.array([1.,0.,1.])\n    expected=np.linalg.solve(fixed.T@np.diag(c)@fixed+.1*np.eye(2),fixed.T@(c*p))\n    assert np.allclose(actual[0],expected,atol=1e-10)\n    assert np.allclose(actual[1],0)\n    return {\'dense_normal_equation_max_error\':float(np.max(np.abs(actual[0]-expected))),\'cold_user_zero_factors\':True}\n')


In [ ]:
from pathlib import Path
Path('models.py').write_text('"""Models fit ONLY on the caller\'s training-window interactions and metadata.\n\nPrediction candidates contain IDs only. Future schedule metadata is never used\nto fit content features. Repeats are allowed consistently across all methods.\n"""\nimport time\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.preprocessing import normalize\nfrom als import ImplicitALS\n\nSEED=2026\nATTRIBUTES=[\'tv_show_category\',\'tv_show_genre_1\',\'tv_show_genre_2\',\n            \'tv_show_genre_3\',\'year_of_production\',\'director\',\'actors\']\n\ndef content_tokens(row):\n    """Field-qualified attribute tokens; no item ID or future data feature."""\n    tokens=[]\n    for field in ATTRIBUTES:\n        value=row.get(field)\n        if pd.isna(value) or str(value).strip() in (\'\',\'0\',\'nan\',\'None\'):\n            continue\n        # Preserve full actor/director names; split common list delimiters only.\n        values=str(value).lower().replace(\';\',\',\').split(\',\')\n        for value in values:\n            value=\'_\'.join(value.strip().split())\n            if value:\n                tokens.append(field+\'=\'+value)\n    return tokens\n\nclass TrainingBundle:\n    def __init__(self,interactions,metadata,users):\n        # interactions represent qualified distinct airings, aggregated by user-item.\n        self.users=list(users)\n        self.uindex={u:i for i,u in enumerate(self.users)}\n        self.items=sorted(set(interactions.tv_show_id)-{0})\n        if not self.items:\n            raise ValueError(\'No training interactions\')\n        self.iindex={x:i for i,x in enumerate(self.items)}\n        counts=interactions.groupby([\'user_id\',\'tv_show_id\']).size().rename(\'count\').reset_index()\n        counts=counts[counts.user_id.isin(self.uindex)]\n        self.counts=csr_matrix((counts[\'count\'].astype(np.float32),\n             (counts.user_id.map(self.uindex),counts.tv_show_id.map(self.iindex))),\n             shape=(len(self.users),len(self.items)))\n        self.weights=self.counts.copy()\n        self.weights.data=np.log1p(self.weights.data)\n        self.popularity=np.asarray((self.counts>0).sum(axis=0)).ravel()\n        self.metadata=metadata.drop_duplicates(\'tv_show_id\').set_index(\'tv_show_id\')\n        self.cold=np.asarray(self.counts.sum(axis=1)).ravel()==0\n\n    def candidate_map(self,candidates):\n        c=np.array(sorted(set(map(int,candidates))-{0}),dtype=np.int64)\n        if len(c)<5:\n            raise ValueError(\'Fewer than five candidates\')\n        ci=np.array([self.iindex.get(int(x),-1) for x in c])\n        pop=np.array([self.popularity[i] if i>=0 else 0 for i in ci])\n        return c,ci,pop\n\n    def rank(self,scores,candidates):\n        """Positive scores first; fill missing/zero/negative slots with popularity.\n\n        Exact ties use training popularity then integer ID. Identical fallback\n        for every model; unseen items have popularity=0 then ID tie-break.\n        """\n        c,_,pop=self.candidate_map(candidates)\n        if scores.shape != (len(self.users),len(c)):\n            raise ValueError(\'Unexpected score matrix shape\')\n        fallback=np.lexsort((c,-pop))\n        rankings={}\n        for ui,user in enumerate(self.users):\n            s=scores[ui]\n            if self.cold[ui]:\n                indices=fallback[:5]\n            else:\n                order=np.lexsort((c,-pop,-np.nan_to_num(s,nan=-np.inf)))\n                chosen=order[np.isfinite(s[order]) & (s[order]>0)][:5].tolist()\n                for j in fallback:\n                    if len(chosen)==5:\n                        break\n                    if int(j) not in chosen:\n                        chosen.append(int(j))\n                indices=chosen\n            rankings[user]=[int(c[i]) for i in indices]\n        return rankings\n\n    def popularity_scores(self,candidates):\n        c,ci,pop=self.candidate_map(candidates)\n        return np.broadcast_to(pop,(len(self.users),len(c))).copy()\n\n    def content_scores(self,candidates):\n        # Vocabulary/IDF fit on train-only item attributes, never hold-out metadata.\n        docs=[content_tokens(row) for _,row in self.metadata.iterrows()]\n        vectorizer=TfidfVectorizer(analyzer=lambda x:x,norm=\'l2\',dtype=np.float32)\n        try:\n            vectorizer.fit(docs)\n            features=vectorizer.transform([content_tokens(self.metadata.loc[x]) if x in self.metadata.index else [] for x in self.items])\n        except ValueError as error:\n            if \'empty vocabulary\' not in str(error):\n                raise\n            return np.zeros((len(self.users),len(self.candidate_map(candidates)[0])),np.float32)\n        profiles=normalize(self.weights@features)\n        c,ci,_=self.candidate_map(candidates)\n        scores=np.zeros((len(self.users),len(c)),np.float32)\n        candidate_features=vectorizer.transform([content_tokens(self.metadata.loc[int(x)]) if int(x) in self.metadata.index else [] for x in c])\n        scores=(profiles@candidate_features.T).toarray()\n        self.content_vectorizer=vectorizer\n        return scores\n\n    def als_scores(self,candidates,factors=32,regularization=.1,alpha=20,iterations=20):\n        # implicit expects OBSERVED confidence values, missing entries mean c=1.\n        confidence=self.weights.copy()\n        confidence.data=1+alpha*confidence.data\n        model=ImplicitALS(factors=factors,regularization=regularization,\n             iterations=iterations,random_state=SEED,num_threads=2)\n        model.fit(confidence)\n        c,ci,_=self.candidate_map(candidates)\n        scores=np.zeros((len(self.users),len(c)),np.float32)\n        known=ci>=0\n        scores[:,known]=model.user_factors@model.item_factors[ci[known]].T\n        self.als_model=model\n        return scores\n\ndef reciprocal_rank_fusion(rankings_a,rankings_b,weight=.5,offset=60):\n    """Fuse full candidate rankings supplied by caller; evaluate hybrid separately."""\n    if set(rankings_a)!=set(rankings_b):\n        raise ValueError(\'Cohort mismatch\')\n    result={}\n    for user in rankings_a:\n        score={}\n        for ranking,w in [(rankings_a[user],weight),(rankings_b[user],1-weight)]:\n            for rank,item in enumerate(ranking,1):\n                score[item]=score.get(item,0)+w/(offset+rank)\n        result[user]=sorted(score,key=lambda x:(-score[x],x))[:5]\n    return result\n')


In [ ]:
from pathlib import Path
Path('test_core.py').write_text('"""Synthetic contract checks; these are AGENT checks, not student verification."""\nimport random\nimport unittest\nfrom core import ScheduleIndex,aggregate_watches,ap_at_k,map_at_k,union_length\n\nclass CoreTests(unittest.TestCase):\n    def test_ap_duplicate_empty_and_cutoff(self):\n        self.assertAlmostEqual(ap_at_k([2,4],[2,2,9,4,5]),.75)\n        self.assertEqual(ap_at_k([],[1,2,3,4,5]),0)\n        self.assertEqual(ap_at_k([6],[1,2,3,4,5,6]),0)\n        self.assertEqual(ap_at_k(range(1,11),list(range(1,6))),1)\n\n    def test_fixed_cohort_includes_empty_truth(self):\n        self.assertEqual(map_at_k({1:[2]},{1:[2],2:[3]},[1,2]),.5)\n\n    def test_interval_boundaries_channels_unlabelled(self):\n        idx=ScheduleIndex([(1,0,100,10,\'a\'),(1,100,200,20,\'b\'),\n                           (1,0,300,0,\'unlabelled\'),(2,0,200,30,\'c\')])\n        self.assertEqual(idx.overlaps(1,50,150),[(10,\'a\',50,100,100),(20,\'b\',100,150,100)])\n        self.assertEqual(idx.overlaps(1,200,250),[])\n        self.assertEqual(idx.overlaps(9,0,100),[])\n\n    def test_union_and_temporal_clipping(self):\n        idx=ScheduleIndex([(1,0,200,10,\'a\')])\n        views=[(1,1,20,130),(1,1,40,150)]\n        train=aggregate_watches(idx,views,0,100,60)\n        self.assertEqual(train[0][\'watch_seconds\'],80)\n        self.assertEqual(aggregate_watches(idx,views,100,200,60),[])\n        self.assertEqual(union_length([(1,3),(2,4),(4,5),(10,12)]),6)\n\n    def test_index_matches_brute_force_with_overlapping_schedule(self):\n        rng=random.Random(2026)\n        rows=[]\n        for i in range(300):\n            s=rng.randrange(1000)\n            rows.append((rng.randrange(3),s,s+rng.randrange(1,500),i+1,str(i)))\n        idx=ScheduleIndex(rows)\n        for _ in range(300):\n            c,s=rng.randrange(4),rng.randrange(1500)\n            e=s+rng.randrange(1,400)\n            expected=sorted((item,key,max(s,a),min(e,b),b-a)\n                            for ch,a,b,item,key in rows if ch==c and max(s,a)<min(e,b))\n            self.assertEqual(sorted(idx.overlaps(c,s,e)),expected)\n\nif __name__==\'__main__\':\n    unittest.main()\n')


In [ ]:
import unittest
import test_core
result=unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromModule(test_core))
assert result.wasSuccessful()


In [ ]:
from als import verify_solver
print("ALS solver check:",verify_solver())


## Audit, time-aware interval join, validation, model selection, refit
All learned transformations use historical training data. No private Kaggle score enters model selection.

In [ ]:
"""Full computation runs in Kaggle. Writes aggregate evidence and final CSV."""
import os,sys,json,time,platform,hashlib,importlib.metadata,resource
from pathlib import Path
from collections import defaultdict,Counter
import numpy as np
import pandas as pd
from core import ScheduleIndex,union_length,ap_at_k,map_at_k,validate_submission
from models import TrainingBundle,SEED

START=time.perf_counter()
OUT=Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
FILES={p.name:p for p in Path('/kaggle/input').rglob('*.csv')}
THRESHOLD=300
CLOCK_AFTER=int(pd.Timestamp('2020-03-29 04:00:00').timestamp())
def save(name,obj):
    (OUT/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)))
def stamp(s):
    raw=int(pd.Timestamp(s).timestamp())
    return raw-3600 if raw>=CLOCK_AFTER else raw
def display(s):
    value=int(s)
    if value>=CLOCK_AFTER-3600: value+=3600
    return str(pd.to_datetime(value,unit='s'))
def seconds(s,fmt):
    parsed=pd.to_datetime(s,format=fmt,errors='raise')
    assert parsed.notna().all()
    raw=parsed.astype('int64')//10**9
    return raw-(raw>=CLOCK_AFTER).astype('int64')*3600

manifest={'seed':SEED,'threshold_seconds':THRESHOLD,'python':sys.version,
          'platform':platform.platform(),'cpu_count':os.cpu_count(),
          'cpu_model':next((x.split(':',1)[-1].strip() for x in Path('/proc/cpuinfo').read_text().splitlines() if x.startswith('model name')),'unknown'),
          'versions':{x:importlib.metadata.version(x) for x in ['numpy','pandas','scipy','scikit-learn','numba']},
          'files':{k:{'size':p.stat().st_size} for k,p in FILES.items()},'timings':{},'checks':{}}
template=pd.read_csv(FILES['submission.csv'],dtype=str)
assert list(template)==['user_id','tv_show_id'] and template.user_id.is_unique
cohort=template.user_id.tolist()
schedule=[]; schema={}; catalog=[]
for name in ['export_arh_11-20-final.csv','export_arh_21-30-final.csv','export_arh_31-42-final.csv']:
    f=pd.read_csv(FILES[name],low_memory=False)
    f['start_s']=seconds(f.start_time,'%d.%m.%Y %H:%M:%S')
    f['end_s']=f.start_s+f.duration
    assert f.tv_show_id.notna().all()
    labelled=f.tv_show_id.ne(0)
    schema[name]={'rows':len(f),'nulls':f.isna().sum().to_dict(),'labelled_rows':int(labelled.sum()),
                  'labelled_row_fraction':float(labelled.mean()),'labelled_duration_fraction':float(f.loc[labelled,'duration'].sum()/f.duration.sum()),
                  'nonpositive_duration_rows':int(f.duration.le(0).sum()),
                  'channels':int(f.channel_id.nunique()),'items':int(f.loc[labelled,'tv_show_id'].nunique()),
                  'start':display(f.start_s.min()),'last_start':display(f.start_s.max()),'end':display(f.end_s.max()),
                  'duration_quantiles':f.duration.quantile([0,.1,.5,.9,.99,1]).to_dict()}
    # Verify duration in seconds against adjacent start times (not assumed all contiguous).
    ordered=f.sort_values(['channel_id','start_s'])
    gap=ordered.groupby('channel_id').start_s.shift(-1)-ordered.start_s
    schema[name]['duration_equals_next_start_gap_fraction']=float((gap==ordered.duration).mean())
    f=f[f.duration.gt(0)].copy()
    labelled=f.tv_show_id.ne(0)
    f['source']=name
    catalog.append(f.loc[labelled].copy())
    schedule.append(f[['channel_id','start_s','end_s','tv_show_id','duration','source']])
    del f
all_schedule=pd.concat(schedule,ignore_index=True)
all_catalog=pd.concat(catalog,ignore_index=True)
del schedule,catalog
# ISO week boundaries are derived from the schedule dates, not from viewing outcomes.
valid_source=all_schedule[all_schedule.source.eq('export_arh_21-30-final.csv')]
test_source=all_schedule[all_schedule.source.eq('export_arh_31-42-final.csv')]
cutoff=stamp(pd.Timestamp(display(valid_source.start_s.min())).normalize())
test_start=stamp(pd.Timestamp(display(test_source.start_s.min())).normalize())
train_start=stamp(pd.Timestamp(display(all_schedule.start_s.min())).normalize())
assert train_start<cutoff<test_start
manifest['split']={'train_start':display(train_start),'cutoff':display(cutoff),'validation_end':display(test_start),
                   'cohort_size':len(cohort),'repeats_allowed':True,'empty_truth_AP':0,'candidate_policy':'labelled IDs airing in prediction window',
                   'test_schedule_features':'ID eligibility only; content metadata from historical schedule'}
history=all_schedule[(all_schedule.start_s<test_start)&all_schedule.end_s.gt(train_start)&all_schedule.tv_show_id.ne(0)].copy()
keys=['channel_id','start_s','end_s','tv_show_id']
manifest['checks']['duplicate_labelled_airings']=int(history.duplicated(keys).sum())
history=history.drop_duplicates(keys).reset_index(drop=True)
history['airing_key']=np.arange(len(history))
index=ScheduleIndex(history[['channel_id','start_s','end_s','tv_show_id','airing_key']].itertuples(index=False,name=None))
valid_candidates=set(map(int,all_schedule.loc[(all_schedule.start_s<test_start)&all_schedule.end_s.gt(cutoff),'tv_show_id']))-{0}
test_candidates=set(map(int,test_source.tv_show_id))-{0}
metadata_train=all_catalog[all_catalog.start_s<cutoff].sort_values('start_s')
metadata_full=all_catalog[all_catalog.start_s<test_start].sort_values('start_s')
manifest['split'].update(validation_candidates=len(valid_candidates),test_candidates=len(test_candidates))
save('schema_audit.json',schema)
print('Split',manifest['split'],flush=True)

segments=defaultdict(list)
coverage={'rows':0,'invalid_intervals':0,'duration_mismatch':0,'clipped_view_seconds':0,'labelled_overlap_seconds':0,
          'rows_with_labelled_overlap':0,'rows_outside_history':0,'rows_crossing_cutoff':0,'overlaps':0,'raw_duration_mismatch':0}
examples=[]; view_users=set(); duration_hist=[]; min_view=None; max_view=None
duration_deltas=Counter();mismatch_examples=[]
join_start=time.perf_counter()
for chunk in pd.read_csv(FILES['dataset11-30.csv'],dtype={'user_id':str},chunksize=250000):
    starts=seconds(chunk.start_time,'%Y-%m-%d %H:%M:%S').to_numpy()
    ends=seconds(chunk.stop_time,'%Y-%m-%d %H:%M:%S').to_numpy()
    lengths=chunk.duraton.to_numpy()
    raw_starts=starts+(starts>=CLOCK_AFTER-3600)*3600
    raw_ends=ends+(ends>=CLOCK_AFTER-3600)*3600
    raw_deltas=raw_ends-raw_starts-lengths
    coverage['raw_duration_mismatch']+=int(np.sum(raw_deltas!=0))
    for delta in raw_deltas[raw_deltas!=0]: duration_deltas[int(delta)]+=1
    for pos in np.flatnonzero(raw_deltas)[:max(0,8-len(mismatch_examples))]:
        mismatch_examples.append({'start_time':str(chunk.iloc[pos].start_time),'stop_time':str(chunk.iloc[pos].stop_time),'duraton':int(lengths[pos]),'wallclock_minus_duraton_seconds':int(raw_deltas[pos])})
    coverage['rows']+=len(chunk)
    coverage['duration_mismatch']+=int(np.sum(ends-starts!=lengths))
    coverage['invalid_intervals']+=int(np.sum(ends<=starts))
    min_view=int(starts.min()) if min_view is None else min(min_view,int(starts.min()))
    max_view=int(ends.max()) if max_view is None else max(max_view,int(ends.max()))
    duration_hist.append(lengths)
    for user,ch,start,end,reported_length in zip(chunk.user_id,chunk.vsetv_id,starts,ends,lengths):
        view_users.add(user)
        delta=int(end-start-reported_length)
        if delta or end<=start:
            continue
        a,b=max(int(start),train_start),min(int(end),test_start)
        if b<=a:
            coverage['rows_outside_history']+=1;continue
        coverage['clipped_view_seconds']+=b-a
        coverage['rows_crossing_cutoff']+=int(a<cutoff<b)
        matches=index.overlaps(int(ch),a,b)
        if matches: coverage['rows_with_labelled_overlap']+=1
        coverage['labelled_overlap_seconds']+=union_length([(x[2],x[3]) for x in matches])
        coverage['overlaps']+=len(matches)
        for item,key,left,right,program_seconds in matches:
            if left<cutoff:
                segments[(0,user,item,key,program_seconds)].append((left,min(right,cutoff)))
            if right>cutoff:
                segments[(1,user,item,key,program_seconds)].append((max(left,cutoff),right))
            if len(examples)<8 and right-left>=THRESHOLD:
                examples.append({'user_id':user,'channel_id':int(ch),'view_start':display(start),'view_end':display(end),
                    'airing_key':int(key),'tv_show_id':int(item),'overlap_start':display(left),'overlap_end':display(right),
                    'overlap_seconds':right-left,'programme_seconds':program_seconds,'view_duration':int(end-start)})
    print('Joined view rows',coverage['rows'],flush=True)
records=[];full_records=[]
for (window,user,item,key,length),ints in segments.items():
    watched=union_length(ints)
    full_records.append((user,int(item),int(key),watched,length))
    if watched>=THRESHOLD:
        records.append((window,user,int(item),int(key),watched,length,watched/length))
del segments
interactions=pd.DataFrame(records,columns=['window','user_id','tv_show_id','airing_key','watch_seconds','programme_seconds','watch_fraction'])
del records
train=interactions[interactions.window.eq(0)].copy();valid=interactions[interactions.window.eq(1)].copy()
full_interactions=pd.DataFrame(full_records,columns=['user_id','tv_show_id','airing_key','watch_seconds','programme_seconds'])
full_interactions=full_interactions.groupby(['user_id','tv_show_id','airing_key','programme_seconds'],as_index=False).watch_seconds.sum()
full_interactions=full_interactions[full_interactions.watch_seconds.ge(THRESHOLD)].copy()
del full_records
assert len(train)>0 and len(valid)>0
durations=np.concatenate(duration_hist)
coverage.update(view_users=len(view_users),first_view=display(min_view),last_view_end=display(max_view),
    duration_quantiles={str(q):float(np.quantile(durations,q)) for q in [0,.01,.1,.25,.5,.75,.9,.95,.99,1]},
    view_fraction_at_least_300=float(np.mean(durations>=THRESHOLD)),
    train_interactions=len(train),validation_interactions=len(valid),train_users=int(train.user_id.nunique()),validation_users=int(valid.user_id.nunique()),
    train_items=int(train.tv_show_id.nunique()),validation_items=int(valid.tv_show_id.nunique()),
    labelled_row_coverage=coverage['rows_with_labelled_overlap']/coverage['rows'],
    labelled_seconds_coverage=coverage['labelled_overlap_seconds']/coverage['clipped_view_seconds'],
    qualified_watch_fraction_quantiles=interactions.watch_fraction.quantile([0,.1,.5,.9,1]).to_dict())
coverage['duration_delta_counts']=dict(duration_deltas)
coverage['duration_mismatch_examples']=mismatch_examples
coverage['duration_policy']='Normalize the observed 2020-03-29 clock jump by subtracting 3600 from timestamps at/after 04:00, in BOTH schedule and views. Transition identified within training history; no absolute timezone inferred. Then require stop-start=duraton for every view.'
assert coverage['duration_mismatch']==0
assert interactions.tv_show_id.ne(0).all() and interactions.watch_fraction.le(1).all()
manifest['timings']['join_seconds']=time.perf_counter()-join_start
# Cross-check actual examples against a second, direct interval calculation.
for example in examples:
    row=history.iloc[example['airing_key']]
    a=max(stamp(example['view_start']),int(row.start_s));b=min(stamp(example['view_end']),int(row.end_s),test_start)
    assert b-a==example['overlap_seconds'] and int(row.channel_id)==example['channel_id']
    example.update(airing_start=display(row.start_s),airing_end=display(row.end_s))
manifest['checks']['actual_overlap_examples_verified']=len(examples)
save('coverage.json',coverage);save('join_examples.json',examples)
print('Coverage',json.dumps(coverage,default=str),flush=True)

truth={u:set(map(int,g.tv_show_id)) for u,g in valid.groupby('user_id') if u in set(cohort)}
assert all(items<=valid_candidates for items in truth.values())
users=sorted(set(train.user_id)|set(cohort))
bundle=TrainingBundle(train,metadata_train,users)
results=[];predictions={};score_cache={}
def evaluate(name,scores,elapsed,params):
    all_ranks=bundle.rank(scores,valid_candidates)
    ranks={u:all_ranks[u] for u in cohort}
    assert all(len(r)==len(set(r))==5 and set(r)<=valid_candidates for r in ranks.values())
    values=np.array([ap_at_k(truth.get(u,()),ranks[u]) for u in cohort])
    active=np.array([bool(truth.get(u)) for u in cohort])
    result={'method':name,'map5':float(values.mean()),'active_map5':float(values[active].mean()),'active_users':int(active.sum()),
            'empty_truth_users':int((~active).sum()),'fit_score_seconds':elapsed,'parameters':params}
    assert abs(result['map5']-map_at_k(truth,ranks,cohort))<1e-12
    results.append(result);predictions[name]=ranks;score_cache[name]=scores
    save('validation_results.json',results);print('RESULT',result,flush=True)

for name,fn,params in [
    ('popularity',lambda:bundle.popularity_scores(valid_candidates),{'statistic':'unique training viewers'}),
    ('content',lambda:bundle.content_scores(valid_candidates),{'features':'category, genres, year, director, actors','profile':'log1p airing count','similarity':'cosine'}),
    ('als32',lambda:bundle.als_scores(valid_candidates,factors=32),{'factors':32,'alpha':20,'regularization':.1,'iterations':20}),
    ('als64',lambda:bundle.als_scores(valid_candidates,factors=64),{'factors':64,'alpha':20,'regularization':.1,'iterations':20})]:
    t=time.perf_counter();scores=fn();evaluate(name,scores,time.perf_counter()-t,params)

# A repeat-frequency component tests whether persistent series habits outperform latent similarity.
c,ci,pop=bundle.candidate_map(valid_candidates)
repeat_scores=np.zeros((len(users),len(c)),np.float32); known=ci>=0
repeat_scores[:,known]=bundle.weights[:,ci[known]].toarray()
evaluate('repeat',repeat_scores,0,{'score':'log1p qualified training airing count; same popularity fallback'})
best_als=max(['als32','als64'],key=lambda n:next(r['map5'] for r in results if r['method']==n))
def rank_score(scores):
    # Reciprocal rank scores over all candidates; same deterministic tie-breaks.
    out=np.zeros_like(scores,dtype=np.float32)
    for i,row in enumerate(scores):
        ordered=np.lexsort((c,-pop,-row))
        positive=ordered[np.isfinite(row[ordered]) & (row[ordered]>0)]
        fallback=np.lexsort((c,-pop))
        positive_set=set(map(int,positive))
        order=np.concatenate([positive,np.array([j for j in fallback if int(j) not in positive_set],dtype=int)])
        out[i,order]=1/(60+np.arange(1,len(c)+1))
    return out
als_rr=rank_score(score_cache[best_als]);repeat_rr=rank_score(repeat_scores)
for w in [.25,.5,.75]:
    t=time.perf_counter();fused=w*repeat_rr+(1-w)*als_rr
    evaluate(f'hybrid_{w}',fused,time.perf_counter()-t,{'repeat_weight':w,'als':best_als,'rank_offset':60})
winner=max(results,key=lambda r:r['map5'])
manifest['selected']=winner
manifest['checks']['train_nonzero_items']=len(bundle.items)
manifest['checks']['cold_cohort_users']=sum(bool(bundle.cold[bundle.uindex[u]]) for u in cohort)
manifest['checks']['validation_new_items']=len(valid_candidates-set(bundle.items))
manifest['checks']['training_metadata_only']=True

# Predetermined activity strata; same two users for every method; no cherry-picked hits.
activity=train.groupby('user_id').size().to_dict()
eligible=sorted([u for u in cohort if truth.get(u) and activity.get(u,0)>0],key=lambda u:(activity[u],u))
chosen=[eligible[len(eligible)//4],eligible[(3*len(eligible))//4]]
lookup=all_catalog.sort_values('start_s').drop_duplicates('tv_show_id').set_index('tv_show_id')
def describe(item,user):
    row=lookup.loc[item]
    return {'id':int(item),'title':str(row.tv_show_title),'category':str(row.tv_show_category),'hit':item in truth.get(user,set()),
            'seen_in_training':bool(((train.user_id==user)&(train.tv_show_id==item)).any())}
user_examples=[]
for user in chosen:
    user_examples.append({'user_id':user,'training_airings':int(activity[user]),'actual_ids':sorted(truth[user]),
                         'training_top':train[train.user_id.eq(user)].tv_show_id.value_counts().head(5).to_dict(),
                         'methods':{n:[describe(x,user) for x in ranks[user]] for n,ranks in predictions.items()},
                         'ap5':{n:ap_at_k(truth[user],ranks[user]) for n,ranks in predictions.items()},
                         'validation_evidence':[dict(tv_show_id=int(r.tv_show_id),watch_seconds=int(r.watch_seconds),
                             programme_seconds=int(r.programme_seconds),airing_start=display(history.iloc[int(r.airing_key)].start_s),
                             channel_id=int(history.iloc[int(r.airing_key)].channel_id))
                             for r in valid[valid.user_id.eq(user)].head(5).itertuples()]})
save('user_examples.json',user_examples)
save('validation_predictions.json',predictions)
save('validation_truth.json',{u:sorted(truth.get(u,set())) for u in cohort})
# Select a nontrivial real AP exercise separately, not as the two comparison users.
exercise_users=[u for u in cohort if 0<ap_at_k(truth.get(u,()),predictions[winner['method']][u])<1]
exercise_user=next((u for u in exercise_users if len(truth[u])<=8),exercise_users[0])
save('personal_check_input.json',{'user_id':exercise_user,'method':winner['method'],
    'predictions':predictions[winner['method']][exercise_user],'actual':sorted(truth[exercise_user]),
    'formula':'AP@5=sum_{r=1..5}(hits_in_top_r/r)*hit_r / min(5, number_of_relevant_items); empty truth -> 0',
    'selected_result':winner,'iteration':[r for r in results if r['method'] in ['als32','als64']]})

# Refit only the validation-selected configuration using all permitted history.
t=time.perf_counter()
final_users=sorted(set(full_interactions.user_id)|set(cohort))
full=TrainingBundle(full_interactions,metadata_full,final_users)
name=winner['method'];par=winner['parameters']
if name=='popularity': final_scores=full.popularity_scores(test_candidates)
elif name=='content': final_scores=full.content_scores(test_candidates)
elif name.startswith('als'): final_scores=full.als_scores(test_candidates,factors=par['factors'])
else:
    c,ci,pop=full.candidate_map(test_candidates)
    final_repeat=np.zeros((len(final_users),len(c)),np.float32);known=ci>=0
    final_repeat[:,known]=full.weights[:,ci[known]].toarray()
    if name=='repeat': final_scores=final_repeat
    else:
        factors=int(par['als'].replace('als',''))
        final_als=full.als_scores(test_candidates,factors=factors)
        w=par['repeat_weight']; final_scores=w*rank_score(final_repeat)+(1-w)*rank_score(final_als)
final_ranks=full.rank(final_scores,test_candidates)
submission=template.copy();submission['tv_show_id']=[' '.join(map(str,final_ranks[u])) for u in cohort]
submission.to_csv(OUT/'submission.csv',index=False)
manifest['checks']['submission']=validate_submission(FILES['submission.csv'],OUT/'submission.csv',test_candidates)
manifest['checks']['submission_sha256']=hashlib.sha256((OUT/'submission.csv').read_bytes()).hexdigest()
manifest['timings']['final_refit_predict_seconds']=time.perf_counter()-t
manifest['timings']['total_seconds']=time.perf_counter()-START
manifest['max_rss_kib']=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
save('run_manifest.json',manifest)
save('test_candidates.json',sorted(test_candidates))
print('FINAL SELECTED',name,'VALIDATION MAP@5',winner['map5'],'CSV CHECK',manifest['checks']['submission'],flush=True)
